In [0]:
pip install databricks-labs-dqx

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.labs.dqx.profiler.profiler import DQProfiler
from databricks.labs.dqx.profiler.generator import DQGenerator
from databricks.labs.dqx.profiler.dlt_generator import DQDltGenerator
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient


In [0]:
input_df = spark.read.table("boeing_catalog.boeing_schema.customer_transactions_delta")

# profile input data
ws = WorkspaceClient()
profiler = DQProfiler(ws)
summary_stats, profiles = profiler.profile(input_df)

# generate DQX quality rules/checks
generator = DQGenerator(ws)
checks = generator.generate_dq_rules(profiles)  # with default level "error"

dq_engine = DQEngine(ws)

# save checks in arbitrary workspace location
dq_engine.save_checks_in_workspace_file(checks, workspace_path="/Workspace/Users/swaroop@beesbridge.us/checks.yml")

# save checks in the installation folder specified in the default run config 
# (only works if DQX is installed in the workspace)
# dq_engine.save_checks_in_installation(checks, run_config_name="default")

# generate DLT expectations
dlt_generator = DQDltGenerator(ws)

# dlt_expectations = dlt_generator.generate_dlt_rules(profiles, language="SQL")
# print(dlt_expectations)

# dlt_expectations = dlt_generator.generate_dlt_rules(profiles, language="Python")
# print(dlt_expectations)

dlt_expectations = dlt_generator.generate_dlt_rules(profiles, language="Python_Dict")
print(dlt_expectations)

In [0]:
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient

checks = DQEngine.load_checks_from_local_file("/Workspace/Users/swaroop@beesbridge.us/boeing/dqx-checks.yml")
dq_engine = DQEngine(WorkspaceClient())

input_df = spark.read.table("boeing_catalog.boeing_schema.customer_transactions_delta")

# Option 1: apply quality rules on the dataframe and provide valid and invalid (quarantined) dataframes
valid_df, quarantined_df = dq_engine.apply_checks_by_metadata_and_split(input_df, checks)

# Option 2: apply quality rules on the dataframe and report issues as additional columns (`_warning` and `_error`)
valid_and_quarantined_df = dq_engine.apply_checks_by_metadata(input_df, checks)

In [0]:
quarantined_df.display()